# 02 - Training (Full Image Finetune)

This notebook finetunes MambaIR on the full-image dataset prepared in `01_data_preparation_full_image.ipynb`.

**Pretrained weights:** MambaIR Gaussian Denoising (sigma=25)

**Hardware:** Tested on Google Colab A100 GPU

```
Pretrained MambaIR (ColorDN sigma=25)
            ↓
   Finetune on full-image blur/sharp pairs
            ↓
   Best checkpoint → used in 04_training_crop.ipynb
```

## (Optional) Mount Google Drive

Run this cell only if you are using **Google Colab** and your files are stored on Google Drive.
Skip this cell if you are running locally.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Step 1 - Check GPU

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2 - Install Dependencies

> ⚠️ This step reinstalls PyTorch and Mamba libraries for A100 + CUDA 12.8 compatibility.
> Takes ~5 minutes. Do not interrupt.

In [ ]:
# Reinstall PyTorch for CUDA 12.8 (A100 compatibility)
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.10.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# Install Mamba dependencies
!pip install causal-conv1d
!pip install mamba-ssm


Check the versions of Pytorch, Cuda after applied compatibilty

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 3 - Clone MambaIR Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists('MambaIR'):
    os.system('git clone https://github.com/csguoh/MambaIR.git')
os.chdir('MambaIR')
print('Working directory:', os.getcwd())

In [ ]:
import os
os.system('pip install -q basicsr einops timm')
os.system('pip install -q causal_conv1d==1.0.0')
os.system('pip install -q mamba_ssm==1.0.1')
print('Dependencies installed.')

In [ ]:
!pip install mamba_ssm --no-build-isolation

In [ ]:
import os
os.system('pip install -q -e .')
print('BasicSR installed.')

## Step 4 - Set Paths

```
# Google Colab + Drive example:
# DATASET_DIR     = '/content/drive/MyDrive/your_project/dataset_resize'
# EXPERIMENT_DIR  = '/content/drive/MyDrive/your_project/experiments'
# PRETRAINED_CKPT = '/content/drive/MyDrive/your_project/pretrained/ColorDN_MambaIR_level25.pth'

# Local machine example:
# DATASET_DIR     = '/path/to/dataset_resize'
# EXPERIMENT_DIR  = '/path/to/experiments'
# PRETRAINED_CKPT = '/path/to/pretrained/ColorDN_MambaIR_level25.pth'
```

In [ ]:
# ← SET YOUR PATHS HERE
DATASET_DIR     = 'dataset_path'       # output folder from 01_data_preparation_resize.ipynb
EXPERIMENT_DIR  = 'experiment_path'    # where checkpoints will be saved
PRETRAINED_CKPT = 'pretrained_path'    # path to pretrained MambaIR weights (.pth)

import os
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs('experiments/pretrained_models', exist_ok=True)
print('Paths set.')

## Step 5 - Download Pretrained Weights

Downloads MambaIR Gaussian Denoising (sigma=25) weights from HuggingFace.

> Skip this step if you already have the weights and set `PRETRAINED_CKPT` accordingly.

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

try:
    model_path = hf_hub_download(
        repo_id='csguoh/MambaIR',
        filename='ColorDN_MambaIR_level25.pth',
        local_dir='experiments/pretrained_models'
    )
    PRETRAINED_CKPT = model_path
    print(f'Downloaded: {model_path}')
except Exception as e:
    print(f'Error: {e}')
    print('Download manually from: https://github.com/csguoh/MambaIR')

## Step 6 - Create Training Config

Key training parameters:

| Parameter | Value | Description |
|-----------|-------|-------------|
| `gt_size` | 64 | Training patch size |
| `batch_size_per_gpu` | 2 | Batch size (increase for A100) |
| `dataset_enlarge_ratio` | 10 | Data augmentation multiplier |
| `total_iter` | 8000 | Total training iterations |
| `lr` | 5e-5 | Learning rate (low for finetuning) |
| `val_freq` | 1000 | Validate every N iterations |

In [ ]:
import os
os.makedirs('options/train/deblur', exist_ok=True)

config = (
    'name: MambaIR_Deblur_Finetune\n'
    'model_type: MambaIRModel\n'
    'scale: 1\n'
    'num_gpu: 1\n'
    'manual_seed: 42\n\n'
    'datasets:\n'
    '  train:\n'
    '    name: DeblurTrain\n'
    '    type: PairedImageDataset\n'
    f'    dataroot_gt: {DATASET_DIR}/train/sharp\n'
    f'    dataroot_lq: {DATASET_DIR}/train/blur\n'
    "    filename_tmpl: '{}'\n"
    '    io_backend:\n'
    '      type: disk\n'
    '    gt_size: 64\n'
    '    use_hflip: true\n'
    '    use_rot: true\n'
    '    use_shuffle: true\n'
    '    num_worker_per_gpu: 2\n'
    '    batch_size_per_gpu: 2\n'
    '    dataset_enlarge_ratio: 10\n'
    '    prefetch_mode: ~\n'
    '  val:\n'
    '    name: DeblurVal\n'
    '    type: PairedImageDataset\n'
    f'    dataroot_gt: {DATASET_DIR}/val/sharp\n'
    f'    dataroot_lq: {DATASET_DIR}/val/blur\n'
    '    io_backend:\n'
    '      type: disk\n\n'
    'network_g:\n'
    '  type: MambaIR\n'
    '  upscale: 1\n'
    '  in_chans: 3\n'
    '  img_size: 128\n'
    '  img_range: 1.\n'
    '  d_state: 16\n'
    '  depths: [6, 6, 6, 6, 6, 6]\n'
    '  embed_dim: 180\n'
    '  mlp_ratio: 1.2\n\n'
    'path:\n'
    f'  pretrain_network_g: {PRETRAINED_CKPT}\n'
    '  strict_load_g: true\n'
    '  resume_state: ~\n\n'
    'train:\n'
    '  optim_g:\n'
    '    type: Adam\n'
    '    lr: !!float 5e-5\n'
    '    weight_decay: 0\n'
    '    betas: [0.9, 0.99]\n'
    '  scheduler:\n'
    '    type: CosineAnnealingRestartLR\n'
    '    periods: [8000]\n'
    '    restart_weights: [1]\n'
    '    eta_min: !!float 1e-7\n'
    '  total_iter: 8000\n'
    '  warmup_iter: -1\n'
    '  pixel_opt:\n'
    '    type: L1Loss\n'
    '    loss_weight: 1.0\n'
    '    reduction: mean\n\n'
    'val:\n'
    '  val_freq: !!float 1000\n'
    '  save_img: false\n'
    '  metrics:\n'
    '    psnr:\n'
    '      type: calculate_psnr\n'
    '      crop_border: 0\n'
    '      test_y_channel: false\n'
    '    ssim:\n'
    '      type: calculate_ssim\n'
    '      crop_border: 0\n'
    '      test_y_channel: false\n\n'
    'logger:\n'
    '  print_freq: 100\n'
    '  save_checkpoint_freq: !!float 1000\n'
    '  save_latest_freq: !!float 500\n'
    '  use_tb_logger: true\n'
    '  wandb:\n'
    '    project: ~\n'
    '    resume_id: ~\n\n'
    'dist_params:\n'
    '  backend: nccl\n'
    '  port: 29500\n'
)

with open('options/train/deblur/train_MambaIR_Deblur_Finetune.yml', 'w') as f:
    f.write(config)
print('Config created: options/train/deblur/train_MambaIR_Deblur_Finetune.yml')

## Step 7 - Start Training

> If the session is interrupted, set `resume_state` in the config to the latest `.state` file
> and re-run this cell to continue from where it left off.

> Example: `resume_state: experiments/MambaIR_Deblur_Finetune/training_states/2000.state`

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

!python basicsr/train.py \
    -opt options/train/deblur/train_MambaIR_Deblur_Finetune.yml \
    --launcher none

## Step 8 - Save Checkpoints

Copies all checkpoints to `EXPERIMENT_DIR` for permanent storage.

In [ ]:
import shutil
from pathlib import Path

ckpt_dir = Path('experiments/MambaIR_Deblur_Finetune')
if ckpt_dir.exists():
    shutil.copytree(str(ckpt_dir), EXPERIMENT_DIR, dirs_exist_ok=True)
    ckpts = sorted(ckpt_dir.glob('models/net_g_[0-9]*.pth'))
    print(f'Saved {len(ckpts)} checkpoints to {EXPERIMENT_DIR}:')
    for c in ckpts:
        print(f'  {c.name}')
else:
    print('Checkpoint folder not found.')

## Step 9 - Plot Training Curves

Run after training is complete to visualize Loss, PSNR, and SSIM curves.

In [ ]:
import re, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130
from pathlib import Path

LOG_DIR  = Path('experiments/MambaIR_Deblur_Finetune')
log_files = sorted(LOG_DIR.glob('*.log'))
log_file = log_files[-1] if log_files else None

if log_file is None:
    print('Log files not found.')
else:
    print(f'Reading log: {log_file.name}')
    text = log_file.read_text(encoding='utf-8', errors='ignore')

    train_iters, train_loss = [], []
    val_iters, val_psnr, val_ssim = [], [], []

    current_iter = 0
    temp_psnr = None

    
    for line in text.split('\n'):
        # Iteration find and store
        m_iter = re.search(r'iter:\s*([\d,]+)', line)
        if m_iter:
            current_iter = int(m_iter.group(1).replace(',', ''))

        m_loss = re.search(r'l_pix:\s*([\d.e+\-]+)', line)
        if m_loss:
            train_iters.append(current_iter)
            train_loss.append(float(m_loss.group(1)))

        m_psnr = re.search(r'psnr:\s*([\d.]+)', line)
        if m_psnr:
            temp_psnr = float(m_psnr.group(1))

        m_ssim = re.search(r'ssim:\s*([\d.]+)', line)
        if m_ssim and temp_psnr is not None:
            val_iters.append(current_iter)
            val_psnr.append(temp_psnr)
            val_ssim.append(float(m_ssim.group(1)))
            temp_psnr = None  
            
    if not val_psnr:
        print("Log files not found! Model may not have been tested yet.")
    else:
        print(f"{len(val_psnr)} validation (test) points found. Plotting...\n")

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.patch.set_edgecolor('black')
        fig.patch.set_linewidth(2)

        axes[0, 0].plot(train_iters, train_loss, marker='o', markersize=3, color='#1f77b4')
        axes[0, 0].set_title('Training L1 Loss', fontweight='bold')
        axes[0, 0].set_xlabel('Iteration')
        axes[0, 0].grid(True, alpha=0.5)

        axes[0, 1].plot(val_iters, val_psnr, marker='s', color='#e74c3c', linestyle=':')
        axes[0, 1].set_title('Validation Checkpoints (PSNR Trend)', fontweight='bold')
        axes[0, 1].set_xlabel('Iteration')
        axes[0, 1].grid(True, alpha=0.5)

        axes[1, 0].plot(val_iters, val_psnr, marker='o', markersize=5, color='#FFA500')
        axes[1, 0].set_title(f'Validation PSNR (Best: {max(val_psnr):.2f})', fontweight='bold')
        axes[1, 0].set_xlabel('Iteration')
        axes[1, 0].grid(True, alpha=0.5)

        axes[1, 1].plot(val_iters, val_ssim, marker='o', markersize=5, color='#2ecc71')
        axes[1, 1].set_title(f'Validation SSIM (Best: {max(val_ssim):.4f})', fontweight='bold')
        axes[1, 1].set_xlabel('Iteration')
        axes[1, 1].grid(True, alpha=0.5)

        plt.tight_layout()
        plt.show()

## Training Complete

Checkpoints are saved at `EXPERIMENT_DIR/models/`.

The best checkpoint (highest PSNR) will be used as the starting point
for the second finetuning stage.

Proceed to **03_data_preparation_crop.ipynb**.